# 3 · Verify the FreeIPA trust anchor

## Goal

Check that the embedded **public CA** is a valid CA certificate and its DER SHA-256 digest matches the local fingerprint file.

## Prerequisites

Obtain the expected CA fingerprint independently from your FreeIPA administrator. A matching repository fingerprint alone is **not** independent authentication.

In [ ]:
from pathlib import Path
import os, subprocess, sys
START = Path.cwd().resolve()
ROOT = next((p for p in (START, *START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()), None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+", " ".join(map(str, argv)), flush=True)
    return subprocess.run([str(a) for a in argv], cwd=ROOT, check=True)
print("Source:", ROOT)

## Action 1 · Inspect the checked-in certificate

No private key or one-time enrollment password belongs in the repository.

In [ ]:
CERT = ROOT/"trust/ipa-ca.crt"
FINGERPRINT = ROOT/"trust/ipa-ca.sha256"
assert CERT.is_file() and FINGERPRINT.is_file(), "Import a reviewed public CA first"
print("Certificate:", CERT.relative_to(ROOT))
print("Local recorded DER SHA-256:", FINGERPRINT.read_text().strip())
run("openssl","x509","-in",CERT,"-noout","-subject","-issuer","-dates")

## Action 2 · Check certificate constraints and digest

The shared runtime library verifies DER digest, validity dates and the CA constraint.

In [ ]:
sys.path.insert(0,str(ROOT/"lib"))
from scos.core import verify_ca
local_fingerprint = verify_ca(CERT, FINGERPRINT.read_text())
print("Local certificate check passed:", local_fingerprint)

## Action 3 · Compare with independently verified fingerprint

Enter the fingerprint obtained **out-of-band**, not copied from `trust/ipa-ca.sha256`. This cell does not send the value anywhere.

In [ ]:
INDEPENDENT_DER_SHA256 = ""
if INDEPENDENT_DER_SHA256:
    assert local_fingerprint == INDEPENDENT_DER_SHA256.strip().lower().replace(":",""), "Independent CA fingerprint mismatch"
    print("Independent CA fingerprint matches")
else:
    print("NOT VERIFIED INDEPENDENTLY — obtain the authoritative fingerprint before building")

## Check

A local certificate check passes only the structural and digest tests. **Do not build or distribute images until independent CA verification is complete.** If replacing the CA, use `notebooks/trust.ipynb` and commit the reviewed public files.

## Next

[Build the pinned toolchain](04-builder.ipynb).